# HyDE Demo：LangChain

HyDE（Hypothetical Document Embeddings）先让 LLM 生成一段假设性答案，再用这段答案进行向量检索，最后把检索到的真实文档交给 LLM 回答。

In [1]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader

load_dotenv()
knowledge_path = "../knowledge_db/prompt_engineering"
persist_path = "../vector_db/hyde-langchain"
embedding_model = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0,
)

/opt/miniconda3/envs/hello-rag-new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/ty/0rm93py51dl129n9z16z4c_h0000gn/T/ipykernel_22732/2609938067.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5422.76it/s]


In [ ]:
# 批量读取目录下的 Markdown 文件
loader = DirectoryLoader(
    knowledge_path, # 要读取的目录
    glob="**/*.md", # 文件匹配规则
    loader_cls=UnstructuredMarkdownLoader, # 指定使用哪个 Loader 读取文件
)
documents = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_path,
)
# search_kwargs 是传给向量检索器的搜索参数。
# 返回一个检索器对象 含有最相似的 4 个文档块
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"documents={len(documents)}, chunks={len(chunks)}")

documents=9, chunks=302


## 1. 生成假设答案

假设答案不是最终答案，只用于改善检索查询。

In [ ]:
hyde_prompt = ChatPromptTemplate.from_template(
    "请针对下面的问题写一段可能出现在知识库中的专业答案。"
    "不要说明你不知道，也不要回答问题之外的内容。\n问题：{question}"
)
hyde_chain = hyde_prompt | llm | StrOutputParser()
# StrOutputParser LangChain 的字符串输出解析器，用来把 LLM 返回的消息对象转换成普通字符串
question = "总结文本转换这篇文章的主要观点、方法和示例"
hypothetical_answer = hyde_chain.invoke({"question": question})
print(hypothetical_answer)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


**文本转换（Text Transformation）核心概述**

**主要观点：**
文本转换旨在通过算法与规则，将原始文本数据重构为符合特定语义、格式或应用需求的结构化输出。其核心价值在于提升数据处理效率、增强信息的可读性，并为下游的自然语言处理（NLP）任务提供高质量的输入。该过程强调在转换过程中保持原始信息的语义完整性，同时实现格式的标准化与规范化。

**核心方法：**
1.  **基于规则的转换（Rule-based Transformation）：** 利用正则表达式、模式匹配及预定义的语法规则，对文本进行清洗、替换及格式重排。
2.  **基于统计与机器学习的转换（Statistical/ML-based Transformation）：** 通过训练模型（如序列到序列模型 Seq2Seq、Transformer 架构）学习文本间的映射关系，实现自动化的文本改写、翻译或摘要生成。
3.  **语义映射与实体对齐（Semantic Mapping）：** 利用知识图谱或本体论，将非结构化文本中的实体与属性映射至预定义的结构化模式（Schema）中。

**典型示例：**
*   **格式化转换：** 将非结构化的日志文件或原始文本提取为 JSON、XML 或 CSV 格式，以便于数据库存储与分析。
*   **文本规范化：** 将口语化表达、缩写或拼写错误转换为标准书面语，例如将“u”转换为“you”，或将日期格式统一为“YYYY-MM-DD”。
*   **风格迁移：** 在保持语义不变的前提下，将文本的语气从正式转换为非正式，或将特定领域的专业术语转换为通俗易懂的解释性文本。
*   **数据脱敏：** 通过转换算法识别并替换文本中的敏感信息（如姓名、电话、身份证号），以满足隐私保护要求。


## 2. 用假设答案检索真实文档，再生成最终答案


```python
context = "\n\n".join(doc.page_content for doc in retrieved_docs)
```

equals

```python
context_parts = []

for doc in retrieved_docs:
    context_parts.append(doc.page_content)

context = "\n\n".join(context_parts)
```

retrieved_docs = [
    Document(page_content="第一段文本"),
    Document(page_content="第二段文本"),
]

[
    "第一段文本",
    "第二段文本",
]

context = "第一段文本\n\n第二段文本"

多个文档对象
→ 多个文本字符串
→ 合并成一个 context 字符串

"\n\n" 表示两个换行符，用它拼接可以让不同文档块之间有明显间隔
好处是：
- 提高可读性；
- 让模型更容易区分不同文档块；
- 避免不同段落的句子直接连在一起；
- 更符合 Prompt 中“上下文分段”的结构。

In [4]:
retrieved_docs = retriever.invoke(hypothetical_answer)
context = "\n\n".join(doc.page_content for doc in retrieved_docs)

answer_prompt = ChatPromptTemplate.from_template(
    "只根据下面的上下文回答问题。如果上下文没有答案，请明确说不知道。"
    "不要把上下文中的单个示例当成整篇文章的主题。\n\n"
    "上下文：{context}\n问题：{question}"
)
answer_chain = answer_prompt | llm | StrOutputParser()
answer = answer_chain.invoke({"context": context, "question": question})
print(answer)

print("\n--- 检索来源 ---")
for doc in retrieved_docs:
    print(doc.metadata.get("source"))

根据提供的上下文，关于“文本转换”的主要观点、方法和示例总结如下：

**主要观点：**
*   大语言模型具备强大的文本转换能力，是开发语言类应用的重要基础。
*   文本转换的应用场景广泛，包括多语言翻译、拼写纠正、语法调整、格式转换等。
*   相比传统方法，大语言模型在翻译上更流畅自然；在格式转换上表现出色，且能提供便于代码解析的结构化输出。
*   开发者在使用大语言模型时应承担社会责任，审慎使用相关功能。

**方法：**
*   通过编程调用API接口，利用Prompt（提示词）引导模型完成特定任务。
*   通过在大规模高质量平行语料上进行Fine-Tune（微调），提升模型在翻译等任务中的精准度。
*   通过指定格式（如JSON、HTML）要求模型输出结构化内容，以便在代码中进一步处理。

**示例：**
*   **文本翻译/风格调整：** 将口语化的询问（“小老弟，我小羊……”）转换为正式的商务信函格式。
*   **格式转换：** 将数据在不同格式（如JSON、HTML、XML、Markdown）之间进行相互转化。
*   **结构化输出：** 要求模型生成包含特定键（book_id、title、author、genre）的虚构书籍清单，并以JSON格式返回。

--- 检索来源 ---
../knowledge_db/prompt_engineering/6. 文本转换 Transforming.md
../knowledge_db/prompt_engineering/6. 文本转换 Transforming.md
../knowledge_db/prompt_engineering/2. 提示原则 Guidelines.md
../knowledge_db/prompt_engineering/7. 文本扩展 Expanding.md
